<a href="https://colab.research.google.com/github/kavyasudha12/Gen-AI-Training-Task/blob/main/Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from wordcloud import WordCloud
import seaborn as sns

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data(num_words=10000)

print("Training samples:", len(x_train))
print("Test samples:", len(x_test))

In [ ]:
word_index = keras.datasets.imdb.get_word_index()
reverse_word_index = {value:key for key,value in word_index.items()}

def decode_review(text):
    return " ".join([reverse_word_index.get(i-3,"?") for i in text])

decoded_reviews = [decode_review(x) for x in x_train[:2000]]

In [ ]:
text_data = " ".join(decoded_reviews)

wordcloud = WordCloud(width=800,height=400,background_color="white").generate(text_data)

plt.figure(figsize=(10,5))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Most Frequent Words")
plt.show()


In [ ]:
bow_vectorizer = CountVectorizer(max_features=5000)
X_bow = bow_vectorizer.fit_transform(decoded_reviews)

# -------- TF-IDF --------
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf_vectorizer.fit_transform(decoded_reviews)

y_small = y_train[:2000]

In [ ]:
lr_model = LogisticRegression(max_iter=200)
lr_model.fit(X_tfidf, y_small)

pred_lr = lr_model.predict(X_tfidf)

print("\nLOGISTIC REGRESSION RESULTS")
print("Accuracy:", accuracy_score(y_small,pred_lr))
print(classification_report(y_small,pred_lr))

In [ ]:
cm = confusion_matrix(y_small,pred_lr)

plt.figure(figsize=(5,4))
sns.heatmap(cm,annot=True,fmt='d',cmap="Blues")
plt.title("Confusion Matrix - Logistic Regression")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

In [ ]:
maxlen = 200

x_train_pad = keras.preprocessing.sequence.pad_sequences(x_train,maxlen=maxlen)
x_test_pad = keras.preprocessing.sequence.pad_sequences(x_test,maxlen=maxlen)


In [ ]:
model = keras.Sequential([
    keras.layers.Embedding(10000,128,input_length=maxlen),
    keras.layers.LSTM(64),
    keras.layers.Dense(1,activation='sigmoid')
])

model.summary()

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
history = model.fit(
    x_train_pad,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.2
)

In [ ]:

loss,acc = model.evaluate(x_test_pad,y_test)

print("\nLSTM Test Accuracy:",acc)

In [ ]:
plt.plot(history.history['accuracy'],label="Train Accuracy")
plt.plot(history.history['val_accuracy'],label="Validation Accuracy")
plt.legend()
plt.title("Training vs Validation Accuracy")
plt.show()



In [ ]:
prediction = model.predict(x_test_pad[:1])

print("\nPrediction Score:",prediction)

if prediction > 0.5:
    print("Positive Review")
else:
    print("Negative Review")